#### Data Analysis: Task 4 & Task 5

In [0]:
spark.table("curated.profit_aggregates").limit(20).display()

order_year,category,sub_category,customer_name,profit
2014,Office Supplies,Paper,Dennis Bolton,79.89
2014,Office Supplies,Art,Andy Yotov,1.10
2014,Furniture,Furnishings,Benjamin Patterson,27.56
2014,null,null,Todd Sumrall,2.25
2014,Office Supplies,Art,Dave Hallsten,1.47
2014,Office Supplies,Storage,Bradley Nguyen,17.43
2014,Office Supplies,Paper,Janet Molinari,12.44
2014,Technology,Accessories,Stefania Perrino,14.40
2014,Office Supplies,Art,Bradley Talbott,10.51
2014,Furniture,Tables,Eva Jacobs,-23.49


Task 5: Using SQL output the following aggregates:
- a) Profit by Year
- b) Profit by Year + Product Category
- c) Profit by Customer
- d) Profit by Customer + Year

In [0]:
%sql
select order_year as Year, sum(profit) as Total_Profit from curated.profit_aggregates
group by 1 order by 1

Year,Total_Profit
2014,39197.81
2015,63010.95
2016,64765.26
2017,110600.80


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
select * from
(
    select order_year as Year, coalesce(category, "NA") as Category, 
    sum(profit) as Total_Profit from curated.profit_aggregates
    -- where category is not null
    group by 1,2 order by 1,2
) pivot (
    sum(Total_Profit) AS Total_Profit
    FOR Category IN ("Furniture" AS Furniture, "Office Supplies" AS Office_Supplies, "Technology" AS Technology, "NA" AS NA)
);

Year,Furniture,Office_Supplies,Technology,NA
2014,-5318.98,22500.32,21493.35,523.12
2015,3027.18,24457.25,34943.36,583.16
2016,6806.45,34385.35,23169.02,404.44
2017,3041.54,43799.51,63271.24,488.51


In [0]:
%sql
select customer_name as Customer, sum(profit) as Total_Profit from curated.profit_aggregates
where customer_name is not null
group by 1 order by 2 desc limit 20

Customer,Total_Profit
Frank Hawley,13400.32
Tamara Chand,8981.32
Raymond Buch,6976.36
Sanjit Chand,5757.30
Hunter Lopez,5622.43
Patrick Ryan,5596.20
Adrian Barton,5444.97
Tom Ashbrook,4703.72
Christopher Martinez,3900.03
Penelope Sewall,3183.77


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
select customer_name as Customer, order_year as Year,
sum(profit) as Total_Profit from curated.profit_aggregates
where customer_name is not null
group by 1,2 order by 1,2 limit 20

Customer,Year,Total_Profit
Aaron Bergman,2014,12.82
Aaron Bergman,2016,116.86
Aaron Hawkins,2014,188.28
Aaron Hawkins,2015,136.56
Aaron Hawkins,2016,38.04
Aaron Hawkins,2017,2.34
Aaron Smayling,2014,32.23
Aaron Smayling,2016,92.14
Aaron Smayling,2017,-377.96
Adam Bellavance,2015,8.71


---------- Task Output End ---------

#### Data Quality Issues Analysis

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F
window = Window.partitionBy("order_id", "product_id").orderBy(F.col("order_date").desc())
print("Duplicate Records:")
spark.table("raw.orders").withColumn("rank", F.count(F.lit(1)).over(window)).filter(F.col("rank") == 2).display()
print("Raw Dataset Records Count: ", spark.table("raw.orders").count())
print("Cleansed Dataset Records Count: ", spark.table("enriched.orders").count())

Duplicate Records:


customer_id,discount,order_date,order_id,price,product_id,profit,quantity,row_id,ship_date,ship_mode,rank
SS-20515,0.0,24/7/2015,CA-2015-103135,135.09,OFF-BI-10000069,62.14,9,5106,28/7/2015,Standard Class,2
SS-20515,0.0,24/7/2015,CA-2015-103135,90.06,OFF-BI-10000069,41.43,6,6584,28/7/2015,Standard Class,2
AB-10060,0.0,1/9/2016,CA-2016-129714,49.12,OFF-PA-10001970,23.09,4,4200,3/9/2016,First Class,2
AB-10060,0.0,1/9/2016,CA-2016-129714,24.56,OFF-PA-10001970,11.54,2,8294,3/9/2016,First Class,2
LC-17140,0.0,23/12/2016,CA-2016-137043,286.38,FUR-FU-10003664,83.05,3,5814,25/12/2016,Second Class,2
LC-17140,0.0,23/12/2016,CA-2016-137043,572.76,FUR-FU-10003664,166.1,6,8028,25/12/2016,Second Class,2
SJ-20125,0.0,15/3/2016,CA-2016-140571,319.76,OFF-PA-10001954,147.09,14,1591,19/3/2016,Standard Class,2
SJ-20125,0.0,15/3/2016,CA-2016-140571,45.68,OFF-PA-10001954,21.0,2,3988,19/3/2016,Standard Class,2
LC-16870,0.2,3/12/2017,CA-2017-118017,76.75,TEC-AC-10002006,10.55,6,1103,6/12/2017,Second Class,2
LC-16870,0.2,3/12/2017,CA-2017-118017,102.34,TEC-AC-10002006,14.07,8,8207,6/12/2017,Second Class,2


Raw Dataset Records Count:  9994
Cleansed Dataset Records Count:  9986


In [0]:
%sql
-- Backfill missing customer names by extracting first/last name from the email address
with extracted as (
  select ai_extract(email, '["first_name", "last_name"]', MAP('version', '2.1')) as cust, customer_name, email 
  from enriched.customers where customer_name is null -- only rows with a missing customer name
)
select
  cust:response:first_name:value::string as first_name,
  cust:response:last_name:value::string as last_name, 
  customer_name, 
  email
from extracted

first_name,last_name,customer_name,email
raquel,flores,null,rachelflores224@gmail.com
margaret,nelson,null,margaretnelson922@gmail.com
jeffrey,neal983,null,jeffreyneal983@gmail.com
null,null,null,carloswallace910@gmail.com
courtney,mueller,null,courtneymueller552@gmail.com
null,null,null,tracywolf621@gmail.com
elizabeth,jennings,null,elizabethjennings888@gmail.com
carol,jordan,null,caroljordan852@gmail.com


In [0]:
%sql
-- orders with product_id in enriched.orders whose product_id has no match in enriched.products
SELECT o.product_id
FROM enriched.orders o
LEFT JOIN enriched.products p
       ON o.product_id = p.product_id
WHERE p.product_id IS NULL
GROUP By 1 limit 10

product_id
OFF-FA-10003112
OFF-EN-10001990
FUR-TA-10004086
OFF-SU-10000946
OFF-EN-10001415
OFF-FA-10002975
FUR-FU-10003981
OFF-FA-10002780
OFF-BI-10001922
OFF-EN-10003134
